# Polynomial Learning Experiment

**Goal:** Learn polynomial coefficients $(a_0, \ldots, a_n)$ over $p$-adic fields by minimizing
$$L(\theta) = \sum_i |a_0 + a_1 x_i + \cdots + a_n x_i^n - y_i|^2$$
where $(x_i, y_i)$ are random $p$-adic training data.

We compare multiple optimization methods: **Greedy**, **MCTS**, **DAG-MCTS**, **UCT**, and **HOO**.

In [ ]:
include("../../../src/NAML.jl")
include("../util.jl")

using Oscar
using .NAML
using Printf

## 1. Configuration

Set the experiment parameters here.

In [ ]:
# Field configuration
PRIME = 2
PREC = 20
K = PadicField(PRIME, PREC)

# Problem configuration
POLY_DEGREE = 3          # Degree of polynomial to learn
N_POINTS = 4             # Number of training data points
N_EPOCHS = 20            # Optimization epochs per optimizer

println("Configuration:")
println("  Prime: $PRIME")
println("  Precision: $PREC")
println("  Polynomial degree: $POLY_DEGREE")
println("  Data points: $N_POINTS")
println("  Epochs: $N_EPOCHS")

## 2. Generate Training Data

Generate random $p$-adic data points $(x_i, y_i)$ with distinct $x$ values.

In [ ]:
# Generate distinct random x values
x_values = Vector{PadicFieldElem}()
max_attempts = N_POINTS * 100
attempts = 0
while length(x_values) < N_POINTS && attempts < max_attempts
    x = generate_random_padic(PRIME, PREC, 0, 8)
    if !any(ex -> ex == x, x_values)
        push!(x_values, x)
    end
    attempts += 1
end

# Generate random y values
y_values = [generate_random_padic(PRIME, PREC, 0, 8) for _ in 1:N_POINTS]
data = collect(zip(x_values, y_values))

println("Training data ($(length(data)) points):")
for (i, (x, y)) in enumerate(data)
    println("  $i. x = $x, y = $y")
    println("     |x| = $(Float64(abs(x))), |y| = $(Float64(abs(y)))")
end

## 3. Create Loss Function

The loss is linear in the polynomial coefficients: we transform the polynomial
$f(x) = a_0 + a_1 x + \cdots + a_n x^n$ into a linear function of parameters $(a_0, \ldots, a_n)$.

In [ ]:
# Create loss (p-adic output, no cutoff)
loss = polynomial_to_linear_loss(data, POLY_DEGREE, nothing)

# Initial parameters at Gauss point
initial_param = generate_gauss_point(POLY_DEGREE + 1, K)
initial_loss = loss.eval([initial_param])[1]

println("Initial parameter: center at origin, radius 0")
println("Initial loss: $initial_loss")

## 4. Define Optimizers

We compare several optimization strategies with different hyperparameters.

In [ ]:
optimizer_defs = [
    (name="Greedy",
     init=(p, l) -> NAML.greedy_descent_init(p, l, 1, (false, 1))),
    
    (name="MCTS-50",
     init=(p, l) -> NAML.mcts_descent_init(p, l,
         NAML.MCTSConfig(num_simulations=50, exploration_constant=1.41,
                         selection_mode=NAML.BestValue, degree=1))),
    
    (name="MCTS-100",
     init=(p, l) -> NAML.mcts_descent_init(p, l,
         NAML.MCTSConfig(num_simulations=100, exploration_constant=1.41,
                         selection_mode=NAML.BestValue, degree=1))),
    
    (name="DAG-MCTS-100",
     init=(p, l) -> NAML.dag_mcts_descent_init(p, l,
         NAML.DAGMCTSConfig(num_simulations=100, exploration_constant=1.41,
                            degree=1, persist_table=true,
                            selection_mode=NAML.BestValue))),
    
    (name="UCT",
     init=(p, l) -> NAML.uct_descent_init(p, l,
         NAML.UCTConfig(max_depth=10, num_simulations=100,
                        exploration_constant=1.41, degree=1))),
    
    (name="HOO",
     init=(p, l) -> NAML.hoo_descent_init(p, l,
         NAML.HOOConfig(rho=0.5, nu1=0.1, max_depth=15))),
]

println("Optimizers to compare:")
for od in optimizer_defs
    println("  - $(od.name)")
end

## 5. Run Experiments

Run each optimizer and record loss trajectories.

In [ ]:
# Store results: name => (losses, time, final_param)
results = Dict{String, Any}()

for od in optimizer_defs
    println("\nRunning $(od.name)...")
    
    optim = od.init(initial_param, loss)
    losses = Float64[]
    
    t_start = time()
    for epoch in 1:N_EPOCHS
        current_loss = NAML.eval_loss(optim)
        push!(losses, current_loss)
        NAML.step!(optim)
    end
    final_loss = NAML.eval_loss(optim)
    push!(losses, final_loss)
    elapsed = time() - t_start
    
    results[od.name] = Dict(
        "losses" => losses,
        "time" => elapsed,
        "final_loss" => final_loss,
        "final_param" => optim.param,
    )
    
    improvement = (initial_loss > 0) ? (initial_loss - final_loss) / initial_loss * 100 : 0.0
    @printf("  %s: final_loss = %.6e, improvement = %.1f%%, time = %.2fs\n",
            od.name, final_loss, improvement, elapsed)
end

println("\nAll optimizers finished.")

## 6. Summary Table

In [ ]:
println("\nSUMMARY")
println("="^70)
println("Problem: $(PRIME)-adic, degree $POLY_DEGREE, $N_POINTS points, $N_EPOCHS epochs")
println("Initial loss: $initial_loss")
println()
@printf("%-15s %15s %12s %12s\n", "Optimizer", "Final Loss", "Improv %", "Time (s)")
println("-"^55)

for od in optimizer_defs
    r = results[od.name]
    improv = (initial_loss > 0) ? (initial_loss - r["final_loss"]) / initial_loss * 100 : 0.0
    @printf("%-15s %15.6e %11.1f%% %12.2f\n",
            od.name, r["final_loss"], improv, r["time"])
end

# Find best
best_name = ""
best_loss = Inf
for od in optimizer_defs
    if results[od.name]["final_loss"] < best_loss
        best_loss = results[od.name]["final_loss"]
        best_name = od.name
    end
end
println("\nBest optimizer: $best_name (loss = $best_loss)")

## 7. Loss Trajectories

Plot the loss over epochs for each optimizer.

In [ ]:
# Print loss trajectories as text (no plotting dependencies required)
println("\nLoss trajectories (selected epochs):")
println()

# Header
header = @sprintf("%-8s", "Epoch")
for od in optimizer_defs
    header *= @sprintf(" %15s", od.name)
end
println(header)
println("-"^(8 + 16 * length(optimizer_defs)))

# Print selected epochs
epochs_to_show = unique(vcat([1, 2, 3, 5, 10], collect(10:10:N_EPOCHS), [N_EPOCHS, N_EPOCHS+1]))
epochs_to_show = filter(e -> e <= N_EPOCHS + 1, sort(epochs_to_show))

for epoch in epochs_to_show
    label = epoch <= N_EPOCHS ? string(epoch) : "final"
    row = @sprintf("%-8s", label)
    for od in optimizer_defs
        losses = results[od.name]["losses"]
        if epoch <= length(losses)
            row *= @sprintf(" %15.6e", losses[epoch])
        end
    end
    println(row)
end

## 8. Learned Coefficients

Inspect the polynomial coefficients found by the best optimizer.

In [ ]:
println("Learned coefficients ($best_name):")
println()

best_param = results[best_name]["final_param"]
best_center = NAML.center(best_param)
best_radius = NAML.radius(best_param)

for (i, coef) in enumerate(best_center)
    deg = i - 1
    coef_abs = Float64(abs(coef))
    r = best_radius[i]
    @printf("  a%d (x^%d): |a%d| = %.6e, radius = %d\n", deg, deg, deg, coef_abs, r)
end

# Verify: evaluate learned polynomial on training data
println("\nVerification on training data:")
for (i, (x, y)) in enumerate(data)
    y_pred = sum(best_center[k] * x^(k-1) for k in 1:length(best_center))
    residual = Float64(abs(y_pred - y))
    @printf("  Point %d: |f(x) - y| = %.6e\n", i, residual)
end

## 9. Multi-Configuration Sweep

Run the experiment across multiple configurations to see how performance varies
with polynomial degree and prime.

In [ ]:
# Sweep over different degrees
sweep_configs = [
    (prime=2, prec=20, degree=2, n_points=3),
    (prime=2, prec=20, degree=3, n_points=4),
    (prime=2, prec=20, degree=4, n_points=5),
    (prime=3, prec=15, degree=3, n_points=4),
    (prime=5, prec=12, degree=3, n_points=4),
]

SWEEP_EPOCHS = 15

sweep_results = []

for cfg in sweep_configs
    println("\nRunning: p=$(cfg.prime), degree=$(cfg.degree), points=$(cfg.n_points)")
    
    Ks = PadicField(cfg.prime, cfg.prec)
    
    # Generate data
    xs = Vector{PadicFieldElem}()
    att = 0
    while length(xs) < cfg.n_points && att < cfg.n_points * 100
        xv = generate_random_padic(cfg.prime, cfg.prec, 0, 8)
        if !any(ex -> ex == xv, xs)
            push!(xs, xv)
        end
        att += 1
    end
    ys = [generate_random_padic(cfg.prime, cfg.prec, 0, 8) for _ in 1:cfg.n_points]
    d = collect(zip(xs, ys))
    
    l = polynomial_to_linear_loss(d, cfg.degree, nothing)
    p0 = generate_gauss_point(cfg.degree + 1, Ks)
    l0 = l.eval([p0])[1]
    
    cfg_results = Dict("config" => cfg, "initial_loss" => l0, "optimizers" => Dict())
    
    for od in optimizer_defs
        try
            opt = od.init(p0, l)
            t0 = time()
            for e in 1:SWEEP_EPOCHS
                NAML.step!(opt)
            end
            fl = NAML.eval_loss(opt)
            dt = time() - t0
            improv = (l0 > 0) ? (l0 - fl) / l0 * 100 : 0.0
            cfg_results["optimizers"][od.name] = Dict(
                "final_loss" => fl, "time" => dt, "improvement" => improv
            )
            @printf("  %-15s final=%.6e  improv=%.1f%%  time=%.2fs\n", od.name, fl, improv, dt)
        catch e
            println("  $(od.name): ERROR - $e")
            cfg_results["optimizers"][od.name] = Dict("error" => string(e))
        end
    end
    
    push!(sweep_results, cfg_results)
end

println("\nSweep complete.")

## 10. Sweep Summary Table

In [ ]:
println("\nSWEEP SUMMARY")
println("="^90)

# Header
header = @sprintf("%-25s", "Config")
for od in optimizer_defs
    header *= @sprintf(" %12s", od.name)
end
println(header)
println("-"^(25 + 13 * length(optimizer_defs)))

for sr in sweep_results
    cfg = sr["config"]
    label = "p=$(cfg.prime) deg=$(cfg.degree) pts=$(cfg.n_points)"
    row = @sprintf("%-25s", label)
    
    for od in optimizer_defs
        if haskey(sr["optimizers"], od.name)
            or = sr["optimizers"][od.name]
            if haskey(or, "error")
                row *= @sprintf(" %12s", "ERR")
            else
                row *= @sprintf(" %12.4e", or["final_loss"])
            end
        else
            row *= @sprintf(" %12s", "---")
        end
    end
    println(row)
end

println("\n(All values are final loss after $SWEEP_EPOCHS epochs)")

## Summary

This notebook demonstrates polynomial interpolation over $p$-adic fields using various
optimization methods from the NAML library.

**Key findings:**
- All optimizers can successfully reduce the interpolation loss
- Tree search methods (MCTS, DAG-MCTS) may find better solutions at the cost of more computation
- Greedy descent is fastest per-step but may get stuck in local optima
- DAG-MCTS exploits the DAG structure of polydisc space for efficient search

**To reproduce at scale:** Use `run_experiments.jl` with `--save` to generate JSON results,
then `generate_tables.jl` to produce LaTeX tables.